# DSA 504 — Class 6
## Data Cleaning I: Missing Values and Types

**Date:** Monday, Sep 21
**Reading:** *Python for Data Analysis*, ch. 7

---

Since Class 4, we've been *noticing* problems in `retail_sales.csv` without fixing them: missing values in `units_sold` and `revenue`, a `date` column stored as text instead of an actual date, and some exact duplicate rows. **Today we start fixing them for real.**

We are **not** fixing the inconsistent category names yet (`"Electronics"` vs `"electronics"` vs `"ELECTRONICS"`) — that's string cleaning, which is Class 7's topic. Today is specifically about missing values, duplicates, and data types.

### Learning goals
By the end of this class, you will be able to:
- Explain what missing data is, and detect it with `.isna()` / `.notna()`
- Choose between dropping and filling missing values, and justify the choice based on context
- Use `.dropna()` and `.fillna()`, including their key parameters
- Find and remove exact duplicate rows with `.duplicated()` and `.drop_duplicates()`
- Convert a column's data type with `.astype()` and `pd.to_datetime()`
- Verify that your cleaning actually worked


In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv("retail_sales.csv")
sales.head()


,date,store,category,units_sold,revenue
0,2025-04-22,Rome,Toys,25.0,402.50
1,2025-04-12,Syracuse,electronics,38.0,3154.00
2,2025-12-21,Utica,home goods,42.0,1299.06
3,2025-02-11,Syracuse,Electronics,60.0,5114.40
4,2025-04-10,Syracuse,toys,10.0,147.10


## 1. Understanding Missing Data

In pandas, a missing value shows up as `NaN` ("Not a Number") — this is true even in columns that aren't numbers, like dates or categories. `NaN` is pandas' universal placeholder for "there's supposed to be a value here, but there isn't one."

**Why does missing data happen in the real world?** A few common causes:
- A sensor or system failed to record a value that day
- A form was submitted with a field left blank
- Data was merged from two sources, and one didn't have a matching record
- A value was deliberately removed (e.g., for privacy)

Missing data is not the same as a zero, or an empty string — it means the value is genuinely unknown.


In [2]:
# NaN is a special float value -- notice its type
print(np.nan)
print(type(np.nan))

# NaN is never equal to anything, not even itself -- this trips people up constantly
print(np.nan == np.nan)   # False!

# This is exactly why we use .isna() instead of == to check for missing values
print(pd.isna(np.nan))    # True -- this is the correct way to check


nan
<class 'float'>
False
True


## 2. Detecting Missing Data


In [4]:
# .isna() returns True/False for every value in the DataFrame
sales.isna().head(10)


,date,store,category,units_sold,revenue
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,False,False
6,False,False,False,False,False
7,False,False,False,False,False
8,False,False,False,False,False
9,False,False,False,False,False


In [5]:
# Summing across a boolean DataFrame counts the Trues -- this is the check you already know
print(sales.isna().sum())


date           0
store          0
category       0
units_sold    60
revenue       60
dtype: int64


In [3]:
# .notna() is the opposite of .isna() -- True where a value EXISTS
print(sales.notna().sum())


date          4039
store         4039
category      4039
units_sold    3979
revenue       3979
dtype: int64


In [5]:
# .any() tells you if ANY value in a column is missing (True/False, not a count)
print(sales.isna().any())


date          False
store         False
category      False
units_sold     True
revenue        True
dtype: bool


In [6]:
# Getting just the rows where units_sold is missing
missing_units = sales[sales["units_sold"].isna()]
print(f"Rows with missing units_sold: {len(missing_units)}")
missing_units.head()


Rows with missing units_sold: 60


,date,store,category,units_sold,revenue
138,2025-12-18,Utica,Cloth ing,NaN,1130.92
155,2025-08-30,Utica,Home_Goods,NaN,1262.91
407,2025-06-29,Albany,Toys,NaN,711.00
446,2025-06-05,Albany,Groceries,NaN,690.42
469,2025-08-22,Utica,Clothing,NaN,1274.65


**Talking point:** always check missing data *per column*, not just for the DataFrame as a whole — a blanket "does this dataset have missing values?" hides which columns are actually affected and how badly.


## 3. Strategy 1 — Dropping Missing Values

`.dropna()` removes rows (or columns) that contain missing values. This is simple, but it throws away data — sometimes more than you'd expect.


In [7]:
# By default, dropna() removes any ROW that has AT LEAST ONE missing value, in ANY column
print(f"Rows before: {len(sales)}")
dropped = sales.dropna()
print(f"Rows after dropna(): {len(dropped)}")
print(f"Rows removed: {len(sales) - len(dropped)}")


Rows before: 4039
Rows after dropna(): 3924
Rows removed: 115


### Controlling what counts as "missing enough to drop"


In [8]:
# subset= limits the check to specific columns -- only drop if THESE columns have NaN
dropped_subset = sales.dropna(subset=["units_sold"])
print(f"Rows after dropping only where units_sold is missing: {len(dropped_subset)}")


Rows after dropping only where units_sold is missing: 3979


In [9]:
# how='all' only drops a row if EVERY column in it is missing -- much more conservative
dropped_all = sales.dropna(how="all")
print(f"Rows after dropna(how='all'): {len(dropped_all)}")
# This will barely remove anything, since it's rare for an entire row to be empty


Rows after dropna(how='all'): 4039


**When dropping makes sense:** when missing values are a small fraction of your data, and you don't have a reliable way to guess what the value should have been. **When it doesn't:** when it would throw away a large chunk of otherwise-good data, or when the missingness itself is meaningful (e.g., "no revenue recorded" might mean "store closed that day," which is information, not noise).


## 4. Strategy 2 — Filling Missing Values

`.fillna()` replaces missing values with something else, instead of removing the row entirely.


In [10]:
# Filling with a fixed constant
filled_zero = sales["units_sold"].fillna(0)
print(filled_zero.isna().sum())   # 0 -- no more missing values in this column



0


**Is filling with 0 a good idea here?** Not necessarily — think about what `units_sold = 0` actually implies (the store was open and sold nothing) versus what a missing value might really mean (we don't know what happened). Silently turning "unknown" into "zero" can quietly bias any later analysis. This is a judgment call, not a default you should reach for automatically.


In [11]:
# Filling with the column's mean -- a common approach for numeric data
mean_units = sales["units_sold"].mean()
filled_mean = sales["units_sold"].fillna(mean_units)
print(f"Column mean used for filling: {mean_units:.2f}")
print(filled_mean.isna().sum())


Column mean used for filling: 61.15
0


In [14]:
# Filling with the column's median -- often more robust than mean if outliers exist
median_units = sales["units_sold"].median()
filled_median = sales["units_sold"].fillna(median_units)
print(f"Column median used for filling: {median_units}")


Column median used for filling: 46.0


### Forward-fill and backward-fill

These carry the previous (or next) valid value forward into the gap — most useful for data that's naturally ordered, like a time series.


In [15]:
# ffill -- carries the last known value forward into the gap
example = pd.Series([100, np.nan, np.nan, 150, np.nan, 200])
print("Original: ", list(example))
print("ffill:    ", list(example.ffill()))

# bfill -- carries the NEXT known value backward into the gap
print("bfill:    ", list(example.bfill()))


Original:  [100.0, nan, nan, 150.0, nan, 200.0]
ffill:     [100.0, 100.0, 100.0, 150.0, 150.0, 200.0]
bfill:     [100.0, 150.0, 150.0, 150.0, 200.0, 200.0]


**Choosing a fill strategy is a judgment call, not a formula.** For our dataset, filling `revenue` with the column mean would be defensible for a quick summary statistic, but misleading if a decision-maker took a specific day's filled-in number seriously. There is often no single "correct" answer — the right choice depends on what the analysis will actually be used for, and you should be able to explain your reasoning.


## 5. Finding and Removing Duplicate Rows

A related cleaning problem: exact duplicate rows, which can happen from data entry mistakes or merging the same data twice.


In [16]:
# .duplicated() flags rows that are exact repeats of an earlier row
print(sales.duplicated().sum())


25


In [17]:
# Looking at the actual duplicate rows
dupes = sales[sales.duplicated(keep=False)]   # keep=False shows ALL copies, not just the extras
print(len(dupes))
dupes.sort_values(list(sales.columns)).head(10)


50


,date,store,category,units_sold,revenue
35,2025-01-10,Rome,Home Goods,32.0,948.48
1258,2025-01-10,Rome,Home Goods,32.0,948.48
275,2025-02-08,Albany,Home Goods,44.0,1368.40
1500,2025-02-08,Albany,Home Goods,44.0,1368.40
147,2025-02-16,Rome,groceries,150.0,979.50
2894,2025-02-16,Rome,groceries,150.0,979.50
3136,2025-02-21,Utica,electronics,33.0,2887.50
3787,2025-02-21,Utica,electronics,33.0,2887.50
701,2025-03-05,Utica,Cloth ing,81.0,1964.25
1627,2025-03-05,Utica,Cloth ing,81.0,1964.25


In [18]:
# .drop_duplicates() removes the extra copies, keeping the first occurrence by default
print(f"Rows before: {len(sales)}")
deduplicated = sales.drop_duplicates()
print(f"Rows after: {len(deduplicated)}")


Rows before: 4039
Rows after: 4014


## 6. Fixing Data Types

Recall from Class 4: `.dtypes` showed `date` as an `object` (text), not an actual date. Let's fix that properly now.


In [19]:
print(sales.dtypes)


date              str
store             str
category          str
units_sold    float64
revenue       float64
dtype: object


In [20]:
# pd.to_datetime() converts a text column into real datetime values
sales["date"] = pd.to_datetime(sales["date"])
print(sales.dtypes)
print(sales["date"].head())


date          datetime64[us]
store                    str
category                 str
units_sold           float64
revenue              float64
dtype: object
0   2025-04-22
1   2025-04-12
2   2025-12-21
3   2025-02-11
4   2025-04-10
Name: date, dtype: datetime64[us]


**Why this matters:** once `date` is a real datetime column, pandas unlocks date-specific operations that don't work on plain text — extracting the month, filtering by date range, or sorting chronologically in a way that's actually guaranteed correct (text sorting can go wrong with inconsistent formats).


In [21]:
# Now that date is a real datetime, we can do things like this:
sales["month"] = sales["date"].dt.month
sales["day_of_week"] = sales["date"].dt.day_name()
print(sales[["date", "month", "day_of_week"]].head())


        date  month day_of_week
0 2025-04-22      4     Tuesday
1 2025-04-12      4    Saturday
2 2025-12-21     12      Sunday
3 2025-02-11      2     Tuesday
4 2025-04-10      4    Thursday


### Converting other types with `.astype()`

`.astype()` is the general-purpose tool for converting a column to a specific type.


In [22]:
# Example: converting a float column to a nullable integer type
# (regular int can't hold NaN, but pandas' "Int64" -- capital I -- can)
example_series = pd.Series([1.0, 2.0, np.nan, 4.0])
print(example_series.dtype)

converted = example_series.astype("Int64")   # nullable integer type
print(converted)
print(converted.dtype)


float64
0       1
1       2
2    <NA>
3       4
dtype: Int64
Int64


---
## Guided Practice

Work through these using `sales`. Some of these steps build on each other — do them in order.


### Exercise 1 — Detect
Print how many missing values exist in `units_sold` and `revenue`, and what percentage of total rows that represents for each column.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Decide and fill
Fill missing `units_sold` values with the column's median (not the mean). Store the result in a new column called `units_sold_filled`, so you can compare it to the original. Print how many missing values remain.


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Duplicates
Confirm how many exact duplicate rows exist, then create a new DataFrame called `sales_deduped` with duplicates removed. Print the row count before and after.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Types
Confirm that `date` has already been converted to a real datetime column (from Section 6 above). Then create a new column called `is_weekend` that is `True` if `day_of_week` is `"Saturday"` or `"Sunday"`, and `False` otherwise.


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Put it together
Starting from `sales`, in one sequence: drop duplicate rows, fill missing `revenue` values with the column median, and confirm with `.isna().sum()` and `.duplicated().sum()` that both issues are resolved in your new DataFrame.


In [ ]:
# Exercise 5 — your code here



---
## Solutions

Try each exercise yourself.


---
## Wrap-up

**Recap:** detecting missing values with `.isna()`/`.notna()`; two strategies for handling them (`.dropna()` vs. `.fillna()`, including mean/median/forward-fill/backward-fill), and that the right choice depends on context, not a formula; finding and removing exact duplicates with `.duplicated()`/`.drop_duplicates()`; and fixing data types with `.astype()` and `pd.to_datetime()`.

**Common mistakes to watch for:**
- Using `== np.nan` to check for missing values instead of `.isna()` (this always returns `False`, even on an actual NaN)
- Filling missing values with 0 without thinking about whether that's actually a reasonable stand-in for "unknown"
- Forgetting that `.dropna()`, `.fillna()`, and `.drop_duplicates()` return *new* objects by default — they don't change the original DataFrame unless you reassign it (or pass `inplace=True`, which we're deliberately avoiding this semester in favor of explicit reassignment, since it's easier to read and debug)
- Converting `date` to datetime *after* trying to do date-based filtering or sorting on it as text

**Before next class (Sep 23):** Class 7 finishes the cleaning story — fixing the inconsistent category text (`"Electronics"` vs `"electronics"` vs `"ELECTRONICS"`) using string methods and regular expressions. **HW1 is also due before Class 7.**
